# Оплата праці медичних працівників України

## Мета дослідження

Проаналізувати рівень зарплат лікарів різних спеціальностей та медичних сестер на основі даних про вакансії.

## Джерело даних

Державна служба зайнятості України. Дані за січень–вересень 2026 року.

In [50]:
import pandas as pd
from pathlib import Path

files = list(Path(".").glob("*.xlsx"))

for file in files:
    print(file.name)

4-sichen-veresen-2026.xlsx


In [51]:
file_path = files[0]

xls = pd.ExcelFile(file_path)

print(xls.sheet_names)

['Vacancies']


C:\Users\1\anaconda3\envs\py311\Lib\site-packages\openpyxl\reader\workbook.py:118: UserWarning: Print area cannot be set to Defined name: Vacancies!$A:$N.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")


In [52]:
df = pd.read_excel(file_path, sheet_name="Vacancies")

print(df.columns.tolist())

['the name of the profession', 'profession code', 'Number of vacancies \nat the end of the period, \nexcluding data on military personnel\n(according to \nForm No. 3-PN)', 'of them, with the minimum wage', 'from the minimum wage to UAH 9,000', 'from 9,000 to 10,000 UAH', 'from 10,000 to 11,000 UAH', 'from 11,000 to 12,000 UAH', 'from 12,000 to 13,000 UAH', 'from 13,000 to 14,000 UAH', 'from 14,000 to 15,000 UAH', 'from 15,000 to 20,000 UAH', 'more than UAH 20,000', 'The average amount of the proposed salary, UAH']


In [53]:
#display(df.head(10))

In [54]:
medical = df[
    df['the name of the profession'].str.contains(
        'лікар|медична сестра|медбрат',
        case=False,
        na=False
    )
]
medical= medical[
        ['the name of the profession',
         'profession code',
         'Number of vacancies \nat the end of the period, \nexcluding data on military personnel\n(according to \nForm No. 3-PN)',
         'The average amount of the proposed salary, UAH']
    ]
display(
    medical.head(50)
)


,the name of the profession,profession code,"Number of vacancies \nat the end of the period, \nexcluding data on military personnel\n(according to \nForm No. 3-PN)","The average amount of the proposed salary, UAH"
6,Лікар-гінеколог,0161.2,1,20000
7,Лікар-епідеміолог,0161.2,2,35000
8,Лікар-терапевт,0161.2,2,48000
9,"Лікар (лікар загальної практики, сімейний лікар)",0161.2,2,30000
10,Лікар з фізичної та реабілітаційної медицини,0161.2,1,37000
66,"завідувач ветеринарної клініки (лікарні, полік...",1221.2,6,15175
80,"начальник (директор, завідувач та ін.) лікарні...",1221.2,4,19225
179,головна медична сестра (головний медичний брат),1229.5,2,8874
238,головний ветеринарний лікар,1237.1,2,30000
406,лікар-патологоанатом,2212.2,5,21304


In [55]:
doctors = medical[
    medical['profession code'].astype(str).str.strip() == '2221.2'
].copy()

display(
    doctors[
        ['the name of the profession',
         'profession code',
         'Number of vacancies \nat the end of the period, \nexcluding data on military personnel\n(according to \nForm No. 3-PN)',
         'The average amount of the proposed salary, UAH']
    ]
)
doctors['profession code'] = (
    doctors['profession code']
    .astype(str)
    .str.strip()
)

,the name of the profession,profession code,"Number of vacancies \nat the end of the period, \nexcluding data on military personnel\n(according to \nForm No. 3-PN)","The average amount of the proposed salary, UAH"
424,лікар,2221.2,8,14768
425,лікар-акушер-гінеколог,2221.2,14,20071
426,лікар-алерголог,2221.2,1,20000
427,лікар-анестезіолог,2221.2,60,20685
428,лікар-гастроентеролог,2221.2,8,19448
429,лікар-дерматовенеролог,2221.2,15,16938
430,лікар-інфекціоніст,2221.2,17,20665
431,лікар-методист,2221.2,2,15217
432,лікар-нарколог,2221.2,5,20000
433,лікар-невролог дитячий,2221.2,13,18452


In [56]:
import numpy as np

name_col = 'the name of the profession'

conditions = [
    doctors[name_col].str.contains(
        'лікар загальної практики|сімейний лікар',
        case=False, na=False
    ),
    doctors[name_col].str.contains(
        'лікар-терапевт',
        case=False, na=False
    ),
    doctors[name_col].str.contains(
        'лікар-анестезіолог',
        case=False, na=False
    ),
    doctors[name_col].str.contains(
        'лікар-педіатр',
        case=False, na=False
    ),
    doctors[name_col].str.contains(
        'лікар-хірург|лікар-нейрохірург',
        case=False, na=False
    )
]

groups = [
    'Сімейний лікар',
    'Лікар-терапевт',
    'Лікар-анестезіолог',
    'Лікар-педіатр',
    'Лікар-хірург'
]

doctors['specialty_group'] = np.select(
    conditions,
    groups,
    default='Інші лікарі-спеціалісти'
)

In [57]:
doctors['specialty_group'].value_counts()

specialty_group
Інші лікарі-спеціалісти    34
Лікар-хірург                7
Лікар-педіатр               2
Лікар-анестезіолог          1
Лікар-терапевт              1
Сімейний лікар              1
Name: count, dtype: int64

In [58]:
print(doctors.columns.tolist())

['the name of the profession', 'profession code', 'Number of vacancies \nat the end of the period, \nexcluding data on military personnel\n(according to \nForm No. 3-PN)', 'The average amount of the proposed salary, UAH', 'specialty_group']


In [59]:
doctors.loc[
    doctors['specialty_group'] == 'Інші лікарі-спеціалісти',
    ['the name of the profession', 'Number of vacancies \nat the end of the period, \nexcluding data on military personnel\n(according to \nForm No. 3-PN)']
]

,the name of the profession,"Number of vacancies \nat the end of the period, \nexcluding data on military personnel\n(according to \nForm No. 3-PN)"
424,лікар,8
425,лікар-акушер-гінеколог,14
426,лікар-алерголог,1
428,лікар-гастроентеролог,8
429,лікар-дерматовенеролог,15
430,лікар-інфекціоніст,17
431,лікар-методист,2
432,лікар-нарколог,5
433,лікар-невролог дитячий,13
434,лікар-невропатолог,59


In [60]:
print(doctors.columns.tolist())

['the name of the profession', 'profession code', 'Number of vacancies \nat the end of the period, \nexcluding data on military personnel\n(according to \nForm No. 3-PN)', 'The average amount of the proposed salary, UAH', 'specialty_group']


In [61]:
exclude = [
    'лікар',
    'лікар-методист',
    "лікар пункту охорони здоров'я",
    'лікар приймальної палати (відділення)'
]

doctors_analysis = doctors[
    ~doctors['the name of the profession'].isin(exclude)
].copy()
doctors_analysis['specialty_group'].value_counts()

specialty_group
Інші лікарі-спеціалісти    30
Лікар-хірург                7
Лікар-педіатр               2
Лікар-анестезіолог          1
Лікар-терапевт              1
Сімейний лікар              1
Name: count, dtype: int64

In [62]:
vac_col = 'Number of vacancies \nat the end of the period, \nexcluding data on military personnel\n(according to \nForm No. 3-PN)'

salary_col = 'The average amount of the proposed salary, UAH'

# Перетворюємо дані на числові
doctors_analysis[vac_col] = pd.to_numeric(
    doctors_analysis[vac_col], errors='coerce'
)

doctors_analysis[salary_col] = pd.to_numeric(
    doctors_analysis[salary_col], errors='coerce'
)

# Розрахунок зваженої середньої зарплати
doctors_analysis['salary_x_vacancies'] = (
    doctors_analysis[salary_col] * doctors_analysis[vac_col]
)

summary = doctors_analysis.groupby('specialty_group').agg(
    vacancies=(vac_col, 'sum'),
    salary_total=('salary_x_vacancies', 'sum')
).reset_index()

summary['average_salary'] = (
    summary['salary_total'] / summary['vacancies']
).round(0)

summary = summary.drop(columns='salary_total')

display(summary)

,specialty_group,vacancies,average_salary
0,Інші лікарі-спеціалісти,436,20005.0
1,Лікар-анестезіолог,60,20685.0
2,Лікар-педіатр,76,18832.0
3,Лікар-терапевт,87,22004.0
4,Лікар-хірург,47,20059.0
5,Сімейний лікар,154,20881.0


In [63]:
doctors.to_csv(
    'doctors_all_specialties.csv',
    index=False,
    encoding='utf-8-sig'
)

In [64]:
summary.to_csv(
    'doctors_salary_by_specialty.csv',
    index=False,
    encoding='utf-8-sig'
)

In [65]:
from pathlib import Path
print(Path.cwd())

C:\Users\1\Medical-staff-ukraine
